In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

: 

# Importar Librerias

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import tensorflow
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [ ]:
data = '/kaggle/input/garbage-classification-v2/garbage-dataset'

# Revisión de los datos

In [ ]:
plt.figure(figsize = (18,7))

categorise = os.listdir(data)
for i, category in enumerate(categorise):
    category_path = os.path.join(data, category)
    image = os.listdir(category_path)[0]
    image_path = os.path.join(category_path, image)
    read_image = plt.imread(image_path)
    plt.subplot(2, 5, i+1)
    plt.imshow(read_image)
    plt.title(category)
    plt.axis('off')

plt.show()

# Preparación de los datos

**División de la muesta para entrenamiento, validación y test**

In [ ]:
from sklearn.model_selection import train_test_split
classes = os.listdir(data)

image_paths = []
labels = []

for class_name in classes:
    class_path = os.path.join(data, class_name)
    for fname in os.listdir(class_path):
        image_paths.append(os.path.join(class_path, fname))
        labels.append(class_name)

df = pd.DataFrame({
    'filename': image_paths,
    'class': labels
})

df

In [ ]:
train, temp = train_test_split(df, test_size=0.3, stratify=df['class'], random_state=42)
valid, test = train_test_split(temp, test_size=0.3, stratify=temp['class'], random_state=42)

In [ ]:
test

In [ ]:
valid

**Generación de imágenes**

In [ ]:
train_datagen = ImageDataGenerator(
    rescale = 1./255,
    rotation_range = 15,
    width_shift_range = 0.1,
    height_shift_range = 0.1,
    shear_range = 0.1,
    zoom_range = 0.1,
    horizontal_flip = True,
    brightness_range=(0.8, 1.2),
    channel_shift_range=20.0,
    fill_mode='nearest'


)
test_valid_datagen = ImageDataGenerator(
    rescale = 1./255
)

In [ ]:
train_ds = train_datagen.flow_from_dataframe(
    train,
    x_col='filename',
    y_col='class',
    target_size = (128, 128),
    batch_size = 32,
    seed = 42,
    shuffle = True,
    class_mode = 'categorical'
)

valid_ds = test_valid_datagen.flow_from_dataframe(
    valid,
    x_col='filename',
    y_col='class',
    target_size = (128,128),
    batch_size = 32,
    seed = 42,
    shuffle = True,
    class_mode = 'categorical'
)

test_ds = test_valid_datagen.flow_from_dataframe(
    test,
    x_col='filename',
    y_col='class',
    target_size = (128, 128),
    seed = 42,
    batch_size = 32,
    shuffle = True,
    class_mode = 'categorical'
)

# Definición del Modelo

**Modelo pre-entrenado MobileNetV2**

Es un modelo que entrega mayor precisión

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.optimizers import Adam

base_model = MobileNetV2(
    input_shape=(128, 128, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False 

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(10, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


In [ ]:
pip install pydot graphviz

In [ ]:
from tensorflow.keras.utils import plot_model

plot_model(model, to_file="model_architecture.png", show_shapes=True, show_layer_names=True)

# Entrenamiento y Ajustes del Modelo

**Early stopping**

In [ ]:
callbacks = [
    EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
),
    ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=1,
    min_lr=1e-6
),
]

**Compilación y entrenamiento en 10 épocas**

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=10,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
model.save("modelo_desechos.h5")

# Evaluación del Modelo

**Accuracy vs Loss**

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axs[0].plot(history.history['accuracy'], label='Entrenamiento')
axs[0].plot(history.history['val_accuracy'], label='Validación')
axs[0].set_title('Precisión')
axs[0].set_xlabel('Épocas')
axs[0].set_ylabel('Accuracy')
axs[0].legend()

# Loss
axs[1].plot(history.history['loss'], label='Entrenamiento')
axs[1].plot(history.history['val_loss'], label='Validación')
axs[1].set_title('Pérdida')
axs[1].set_xlabel('Épocas')
axs[1].set_ylabel('Loss')
axs[1].legend()

plt.tight_layout()
plt.show()


**Hacer Predicciones**

In [ ]:
#  Make Predictions on Test Set
print("MAKING PREDICTIONS ON TEST SET")
print("="*50)

# Get predictions
test_predictions = model.predict(test_ds)
test_pred_classes = np.argmax(test_predictions, axis=1)

# Get true labels
test_ds.reset()
true_labels = []
for i in range(len(test_ds)):
    batch_labels = test_ds[i][1]
    true_labels.extend(np.argmax(batch_labels, axis=1))
    if i >= len(test_ds) - 1:
        break

true_labels = np.array(true_labels[:len(test_pred_classes)])

# Get class names
class_names = list(test_ds.class_indices.keys())
print(f"Classes: {class_names}")

#  Display Sample Predictions with Confidence
print("\n" + "="*50)
print("SAMPLE PREDICTIONS WITH CONFIDENCE")
print("="*50)

# Get some test images and predictions
test_ds.reset()
sample_batch = next(iter(test_ds))
sample_images, sample_labels = sample_batch
sample_predictions = model.predict(sample_images)
# Display first 15 predictions
plt.figure(figsize=(16, 12))
for i in range(min(15, len(sample_images))):
    plt.subplot(3, 5, i+1)
    plt.imshow(sample_images[i])

    true_class = class_names[np.argmax(sample_labels[i])]
    pred_class = class_names[np.argmax(sample_predictions[i])]
    confidence = np.max(sample_predictions[i]) * 100

    color = 'green' if true_class == pred_class else 'red'
    plt.title(f'True: {true_class}\nPred: {pred_class}\nConf: {confidence:.1f}%',
              color=color, fontsize=10)
    plt.axis('off')

plt.suptitle('Sample Predictions with Confidence Scores', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Print detailed predictions for first few samples
print("\nDetailed Predictions (First 10 samples):")
for i in range(min(10, len(sample_predictions))):
    true_class = class_names[np.argmax(sample_labels[i])]
    pred_class = class_names[np.argmax(sample_predictions[i])]
    confidence = np.max(sample_predictions[i]) * 100

    status = "✓" if true_class == pred_class else "✗"
    print(f"{status} Sample {i+1:2d}: True={true_class:12s} | Pred={pred_class:12s} | Confidence={confidence:5.1f}%")

**Reporte de Clasificación**

In [ ]:
# Classification Report
print("\n" + "="*50)
print("CLASSIFICATION REPORT")
print("="*50)

try:
    report = classification_report(true_labels, test_pred_classes,
                                 target_names=class_names, digits=3)
    print(report)
except:
    print("sklearn not available, showing manual classification metrics:")

    # Manual classification metrics
    from collections import defaultdict

    metrics = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0})

    for true_label, pred_label in zip(true_labels, test_pred_classes):
        if true_label == pred_label:
            metrics[true_label]['tp'] += 1
        else:
            metrics[pred_label]['fp'] += 1
            metrics[true_label]['fn'] += 1

    print(f"{'Class':<15} {'Precision':<10} {'Recall':<10} {'F1-Score':<10} {'Support':<10}")
    print("-" * 60)

    total_support = 0
    weighted_precision = 0
    weighted_recall = 0
    weighted_f1 = 0

    for i, class_name in enumerate(class_names):
        tp = metrics[i]['tp']
        fp = metrics[i]['fp']
        fn = metrics[i]['fn']
        support = tp + fn

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

        print(f"{class_name:<15} {precision:<10.3f} {recall:<10.3f} {f1:<10.3f} {support:<10}")

        total_support += support
        weighted_precision += precision * support
        weighted_recall += recall * support
        weighted_f1 += f1 * support

    print("-" * 60)
    print(f"{'Weighted Avg':<15} {weighted_precision/total_support:<10.3f} {weighted_recall/total_support:<10.3f} {weighted_f1/total_support:<10.3f} {total_support:<10}")
    print(f"{'Accuracy':<15} {np.mean(true_labels == test_pred_classes):<10.3f}")

**Matriz de Confusión**

In [ ]:
# Confusion Matrix
print("\n" + "="*50)
print("CONFUSION MATRIX")
print("="*50)

try:
    cm = confusion_matrix(true_labels, test_pred_classes)
except:
    print("sklearn not available, creating manual confusion matrix:")

    # Manual confusion matrix
    n_classes = len(class_names)
    cm = np.zeros((n_classes, n_classes), dtype=int)

    for true_label, pred_label in zip(true_labels, test_pred_classes):
        cm[true_label, pred_label] += 1

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix', fontsize=16, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Save the trained model
model.save('garbage_classifier.h5')
print("Model saved successfully!")

# Verify the model works
import numpy as np

# Test with a dummy image (224x224x3)
test_img = np.random.rand(1, 224, 224, 3)
pred = model.predict(test_img)
print(f"Test prediction shape: {pred.shape}")  # Should be (1, 6)
print(f"Predictions sum to 1: {np.sum(pred)}")  # Should be close to 1.0
print(f"Sample prediction: {pred[0]}")